# 10 · Pipeline Patterns

> **Module 8 & 9 · Case Study & Streaming · Lectures 8.1–8.3, 9.1–9.2**

In this notebook you will:

- Build an incremental, idempotent batch pipeline
- Add data-quality gates that fail the run
- Stream from Pub/Sub into BigQuery
- Handle late and duplicate events
- Orchestrate the whole thing with Airflow / Composer

---

## Setup

In [ ]:
import os
import sys
import json
import hashlib
import datetime as dt

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

BUCKET = os.environ.get("COURSE_BUCKET", f"{PROJECT_ID}-bq-course")
print(f"Bucket: {BUCKET}")

## 1. The three layers of a data platform

| Layer | Contents | Write pattern |
|---|---|---|
| **Landing / raw** | Exactly what arrived, unmodified | Append only |
| **Curated** | Conformed, deduplicated, typed | `MERGE` / `REPLACE PARTITION` |
| **Marts** | Aggregations shaped for BI | `CREATE OR REPLACE` |

Every layer is **partitioned by the same business date**, so a backfill of one
day touches exactly one partition in each layer.

In [ ]:
for ds in ("raw", "curated", "mart", "staging", "dq"):
    dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{ds}")
    dataset_ref.location = "US"
    client.create_dataset(dataset_ref, exists_ok=True)
    print(f"✓ {ds}")

# The source table, partitioned by business date
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.raw.orders_raw` (
      order_id      STRING,
      customer_id   STRING,
      order_date    DATE,
      order_ts      TIMESTAMP,
      total_amount  NUMERIC,
      status        STRING,
      region        STRING,
      channel       STRING,
      ingest_hash   STRING,
      ingested_at   TIMESTAMP
    )
    PARTITION BY order_date
    CLUSTER BY customer_id, region
    OPTIONS (
      partition_expiration_days = 365,
      require_partition_filter = TRUE
    )
    """
)
print("✓ raw.orders_raw created")

## 2. Idempotency — the single most important property

A pipeline is idempotent when re-running it for the same day produces the same
result and does not double-count. BigQuery gives you three ways to get there.

In [ ]:
# A content hash lets you detect changed rows without a schema diff
def row_hash(*parts) -> str:
    joined = "|".join(str(p) for p in parts)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:32]


TARGET_DATE = dt.date(2026, 1, 15)

rows = []
for i in range(50):
    d = TARGET_DATE
    order_id = f"o{i:05d}"
    customer_id = f"c{(i % 20) + 1:04d}"
    amount = round(10 + (i * 7.3) % 490, 2)
    status = "complete" if i % 5 else "active"
    region = ["EMEA", "AMER", "APAC"][i % 3]
    channel = ["web", "mobile"][i % 2]
    rows.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_date": d.isoformat(),
        "order_ts": f"{d.isoformat()}T{10 + i % 8:02d}:15:00Z",
        "total_amount": amount,
        "status": status,
        "region": region,
        "channel": channel,
        "ingest_hash": row_hash(order_id, customer_id, amount, status),
        "ingested_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    })

errors = client.insert_rows_json(
    client.dataset("raw").table("orders_raw$20260115"), rows
)
print(f"✓ landed {len(rows)} rows, errors={len(errors) if errors else 0}")

In [ ]:
# ---------- Strategy 1: REPLACE PARTITION (replace the day wholesale) ----------
REPLACE_SQL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_part$20260115` AS
SELECT
  order_id,
  customer_id,
  order_date,
  total_amount,
  status,
  region,
  channel,
  CURRENT_TIMESTAMP() AS curated_at
FROM `{PROJECT_ID}.raw.orders_raw$20260115`
WHERE order_id IS NOT NULL
  AND total_amount IS NOT NULL
"""

run_query(REPLACE_SQL)
print("✓ curated partition replaced")

df = run_to_dataframe(
    f"""
    SELECT COUNT(*) AS rows FROM `{PROJECT_ID}.curated.orders_part$20260115`
    """
)
display(df)

In [ ]:
# ---------- Strategy 2: MERGE (upsert, only for incremental corrections) ----------
MERGE_SQL = f"""
MERGE `{PROJECT_ID}.curated.orders_part` AS t
USING (
  SELECT 'o00007' AS order_id, 'c0008' AS customer_id,
         DATE '2026-01-15' AS order_date, 999.99 AS total_amount,
         'reconciled' AS status, 'EMEA' AS region, 'web' AS channel,
         CURRENT_TIMESTAMP() AS curated_at
) AS s
ON t.order_id = s.order_id AND t.order_date = s.order_date
WHEN MATCHED THEN UPDATE SET
  t.total_amount = s.total_amount,
  t.status = s.status,
  t.curated_at = s.curated_at
WHEN NOT MATCHED THEN INSERT
  (order_id, customer_id, order_date, total_amount, status, region, channel, curated_at)
VALUES (s.order_id, s.customer_id, s.order_date, s.total_amount, s.status, s.region, s.channel, s.curated_at)
"""

job = client.query(MERGE_SQL)
job.result()
print(f"✓ MERGE affected {job.num_dml_affected_rows} rows")

print("Note: MERGE must match on the partition key, or it scans the whole table.")

In [ ]:
# ---------- Strategy 3: INSERT ... SELECT with a NOT EXISTS anti-join ----------
INSERT_SQL = f"""
INSERT INTO `{PROJECT_ID}.curated.orders_part`
  (order_id, customer_id, order_date, total_amount, status, region, channel, curated_at)
SELECT
  r.order_id, r.customer_id, r.order_date, r.total_amount,
  r.status, r.region, r.channel, CURRENT_TIMESTAMP()
FROM `{PROJECT_ID}.raw.orders_raw$20260115` AS r
WHERE r.order_date = DATE '2026-01-15'
  AND NOT EXISTS (
    SELECT 1 FROM `{PROJECT_ID}.curated.orders_part$20260115` AS c
    WHERE c.order_id = r.order_id
  )
"""

job = client.query(INSERT_SQL)
job.result()
print(f"✓ INSERT appended {job.num_dml_affected_rows} new rows")

print("This is the append-only variant — it never updates existing rows.")

## 3. Deduplication

Streaming and batch pipelines both deliver the same event twice. Deduplicate on
a business key, keeping the latest.

In [ ]:
DEDUP_SQL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_dedup` AS
SELECT * EXCEPT(rn)
FROM (
  SELECT
    order_id, customer_id, order_date, total_amount, status, region, channel,
    ingested_at,
    ROW_NUMBER() OVER (
      PARTITION BY order_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM `{PROJECT_ID}.raw.orders_raw$20260115`
)
WHERE rn = 1
"""

run_query(DEDUP_SQL)
print("✓ deduplicated")

df = run_to_dataframe(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{PROJECT_ID}.raw.orders_raw$20260115`) AS raw_rows,
      (SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_dedup`)        AS dedup_rows,
      (SELECT COUNT(DISTINCT order_id) FROM `{PROJECT_ID}.raw.orders_raw$20260115`) AS distinct_ids
    """
)
display(df)

## 4. Data-quality gates

A gate is a **query that returns a pass/fail**. Run it after every transform;
fail the pipeline if it fails.

In [ ]:
DQ_CHECKS = {
    "no null order_id": f"""
        SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_dedup`
        WHERE order_id IS NULL OR order_id = ''
    """,
    "no negative amounts": f"""
        SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_dedup`
        WHERE total_amount < 0
    """,
    "no orphan customers": f"""
        SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_dedup`
        WHERE customer_id IS NULL
    """,
    "row count above threshold": f"""
        SELECT CASE WHEN COUNT(*) >= 10 THEN 0 ELSE 1 END
        FROM `{PROJECT_ID}.curated.orders_dedup`
    """,
    "no duplicate order_ids": f"""
        SELECT COUNT(*) FROM (
          SELECT order_id FROM `{PROJECT_ID}.curated.orders_dedup`
          GROUP BY order_id HAVING COUNT(*) > 1
        )
    """,
}


def run_dq_gates(checks: dict, threshold: int = 0) -> dict:
    """Run each check; return {name: (passed, offending_rows)}."""
    results = {}
    for name, sql in checks.items():
        job = client.query(sql)
        rows = list(job.result())
        offending = int(rows[0][0]) if rows else 0
        passed = offending <= threshold
        results[name] = (passed, offending)
    return results


gate_results = run_dq_gates(DQ_CHECKS)

print(f"{'check':<28} {'result':<8} {'offending':>10}")
print("-" * 48)
for name, (passed, offending) in gate_results.items():
    print(f"{name:<28} {'PASS' if passed else 'FAIL':<8} {offending:>10,}")

In [ ]:
# The Airflow version of the same thing — raising fails the task
AIRFLOW_DQ = '''
def enforce_dq(**context):
    checks = {
        "null_order_id": f"SELECT COUNT(*) FROM {PROJECT}.curated.orders_dedup WHERE order_id IS NULL",
        "negative_amount": f"SELECT COUNT(*) FROM {PROJECT}.curated.orders_dedup WHERE total_amount < 0",
        "row_count": f"SELECT COUNT(*) FROM {PROJECT}.curated.orders_dedup",
    }
    failures = []
    for name, sql in checks.items():
        rows = client.query(sql).result()
        value = int(rows[0][0])
        if name == "row_count":
            ok = value > 0
        else:
            ok = value == 0
        if not ok:
            failures.append(f"{name}={value}")
    if failures:
        raise ValueError(f"DQ gate failed: {', '.join(failures)}")
    return "all checks passed"
'''

print(AIRFLOW_DQ)

## 5. Build the mart

Marts are fully replaced on every run — they are derived, so there is no
reason to keep history of them.

In [ ]:
MART_SQL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.mart.daily_revenue`
PARTITION BY order_date
CLUSTER BY region, channel
AS
SELECT
  order_date,
  region,
  channel,
  COUNT(*)            AS order_count,
  SUM(total_amount)   AS revenue,
  AVG(total_amount)   AS avg_order_value
FROM `{PROJECT_ID}.curated.orders_dedup`
GROUP BY order_date, region, channel
"""

run_query(MART_SQL)
print("✓ mart.daily_revenue rebuilt")

df = run_to_dataframe(
    f"""
    SELECT order_date, region, channel, order_count, revenue
    FROM `{PROJECT_ID}.mart.daily_revenue`
    ORDER BY revenue DESC
    LIMIT 10
    """
)
display(df)

In [ ]:
# A materialized view over the mart — the serving layer for the dashboard
run_query(
    f"""
    CREATE MATERIALIZED VIEW IF NOT EXISTS `{PROJECT_ID}.mart.mv_daily_revenue`
    PARTITION BY order_date
    CLUSTER BY region, channel
    OPTIONS (
      enable_refresh = TRUE,
      max_staleness = INTERVAL 12 HOUR,
      refresh_interval_minutes = 60
    )
    AS
    SELECT
      order_date, region, channel,
      COUNT(*) AS order_count,
      SUM(total_amount) AS revenue
    FROM `{PROJECT_ID}.curated.orders_dedup`
    GROUP BY order_date, region, channel
    """
)
print("✓ MV created")

# Compare the bytes the dashboard query costs against the MV
mart_est = estimate_cost(
    f"SELECT region, SUM(revenue) FROM `{PROJECT_ID}.mart.daily_revenue` "
    "WHERE order_date = DATE '2026-01-15' GROUP BY region"
)
mv_est = estimate_cost(
    f"SELECT region, SUM(revenue) FROM `{PROJECT_ID}.mart.mv_daily_revenue` "
    "WHERE order_date = DATE '2026-01-15' GROUP BY region"
)

print(f"mart: {mart_est['bytes_processed']:>10,} bytes")
print(f"MV:   {mv_est['bytes_processed']:>10,} bytes")

## 6. Streaming into BigQuery

Three routes, in order of simplicity:

1. **Pub/Sub → BigQuery direct push** — no compute, but no transforms.
2. **Pub/Sub → Dataflow → BigQuery** — transforms and windowing.
3. **Pub/Sub → Dataflow → BigLake Iceberg** — open table format.

In [ ]:
# The streaming destination table
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.raw.events_stream` (
      event_id     STRING,
      user_id      STRING,
      event_type   STRING,
      event_ts     TIMESTAMP,
      payload      JSON
    )
    PARTITION BY DATE(event_ts)
    CLUSTER BY event_type, user_id
    OPTIONS (
      partition_expiration_days = 30,
      require_partition_filter = TRUE
      -- Require a row-level INSERT_ID to make the load idempotent
    )
    """
)
print("✓ raw.events_stream created")

print("Streaming inserts are billed at $0.01 per 200 MB, with the first")
print("1 GB per table per day free.")

In [ ]:
# Insert streaming rows with explicit insertIds — the idempotency key
stream_rows = [
    {
        "event_id": "e0001",
        "user_id": "u0001",
        "event_type": "page_view",
        "event_ts": "2026-01-15T10:00:00Z",
        "payload": json.dumps({"path": "/home", "ms": 120}),
    },
    {
        "event_id": "e0002",
        "user_id": "u0002",
        "event_type": "purchase",
        "event_ts": "2026-01-15T10:05:00Z",
        "payload": json.dumps({"amount": 42.5, "currency": "EUR"}),
    },
]

errors = client.insert_rows_json(
    client.dataset("raw").table("events_stream"),
    stream_rows,
    row_ids=[f"{r['event_id']}-20260115" for r in stream_rows],
)
print(f"✓ streamed {len(stream_rows)} rows, errors={errors if errors else 'none'}")

print("Re-running with the same row_ids is a no-op — BigQuery rejects duplicates.")

## 7. Late and duplicate events

Streaming data is never complete at wall-clock time. Handle late arrivals with
a lookback window on every run.

In [ ]:
# The batch job reprocesses the last N days every run — this is the fix
LOOKBACK_DAYS = 3

REPROCESS_SQL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.events_clean`
PARTITION BY event_date
AS
SELECT
  event_id,
  user_id,
  event_type,
  event_ts,
  DATE(event_ts) AS event_date,
  payload,
  CURRENT_TIMESTAMP() AS processed_at
FROM (
  SELECT
    event_id, user_id, event_type, event_ts, payload,
    ROW_NUMBER() OVER (
      PARTITION BY event_id
      ORDER BY event_ts DESC
    ) AS rn
  FROM `{PROJECT_ID}.raw.events_stream`
  WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {LOOKBACK_DAYS} DAY)
)
WHERE rn = 1
"""

run_query(REPROCESS_SQL)
print(f"✓ reprocessed the last {LOOKBACK_DAYS} days")

# How late do events actually arrive?
df = run_to_dataframe(
    f"""
    SELECT
      DATE(event_ts) AS event_date,
      COUNT(*) AS events,
      COUNT(DISTINCT user_id) AS users
    FROM `{PROJECT_ID}.curated.events_clean`
    GROUP BY event_date
    ORDER BY event_date
    """
)
display(df)

## 8. The Airflow DAG

The same pipeline expressed as a DAG. Every task is idempotent, so a retry
is always safe.

In [ ]:
DAG_CODE = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import (
    BigQueryInsertJobOperator, BigQueryCreateEmptyTableOperator,
)
from airflow.providers.google.cloud.sensors.gcs import GCSObjectExistenceSensor
from airflow.operators.python import PythonOperator

PROJECT = "YOUR_PROJECT_ID"
BUCKET = "YOUR_PROJECT_ID-bq-course"

default_args = {
    "owner": "data-eng",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "depends_on_past": False,
}

with DAG(
    dag_id="retailmart_daily",
    default_args=default_args,
    schedule_interval="0 3 * * *",
    catchup=False,
    max_active_runs=1,
    dagrun_timeout=timedelta(hours=2),
    tags=["bigquery", "retailmart"],
) as dag:

    def resolve_target_date(**context):
        # data_interval_start is the business date being processed
        return (context["data_interval_start"] - timedelta(days=1)).strftime("%Y%m%d")

    resolve = PythonOperator(task_id="resolve_target_date", python_callable=resolve_target_date)

    wait_for_file = GCSObjectExistenceSensor(
        task_id="wait_for_pos_file",
        bucket=BUCKET,
        object_name="landing/{{ ds }}/pos_orders.csv",
        poke_interval=60,
        timeout=3600,
        mode="reschedule",   # frees the worker slot while it waits
    )

    land = BigQueryCreateEmptyTableOperator(
        task_id="land_pos_file",
        dataset_id=f"{PROJECT}.raw",
        table_id="orders_raw",
        schema_fields=[...],
        time_partitioning={"type": "DAY", "field": "order_date"},
        clustering_fields=["customer_id", "region"],
    )

    conform = BigQueryInsertJobOperator(
        task_id="conform_orders",
        configuration={"query": {"query": "...REPLACE PARTITION..."}},
    )

    dq = PythonOperator(task_id="dq_checks", python_callable=run_dq_gates_task)

    publish = BigQueryInsertJobOperator(
        task_id="publish_marts",
        configuration={"query": {"query": "...CREATE OR REPLACE MART..."}},
    )

    resolve >> wait_for_file >> land >> conform >> dq >> publish
'''

print(DAG_CODE)

In [ ]:
# Backfill: re-run the DAG for a range of past dates
print("""
Backfill one day:
  airflow dags backfill retailmart_daily -s 2026-01-14 -e 2026-01-14

Backfill a range:
  airflow dags backfill retailmart_daily -s 2026-01-01 -e 2026-01-14

On Cloud Composer:
  gcloud composer environments run-airflow-dag \
    --environment=retailmart-composer \
    --dag-id=retailmart_daily \
    --run-date=2026-01-14T00:00:00Z
    """)

## 9. The end-to-end lineage

```
GCS landing/            raw.orders_raw        curated.orders_part
  pos_orders.csv    ->   (append only)    ->   (REPLACE PARTITION)
                              |                      |
                              |                      v
                              |               curated.orders_dedup
                              |                      |
                              |                      v
                              |               mart.daily_revenue
                              |                      |
                              |                      v
                              |               mart.mv_daily_revenue
                              |                      |
                              |                      v
Pub/Sub events      ->   raw.events_stream ->  Looker Studio dashboard
  (streaming)            (append, row_ids)      (BI Engine optional)
```

Every arrow is a **query you can re-run**. Every table is **partitioned by the
same business date**. Every step is **idempotent**.

In [ ]:
# Trace the lineage programmatically
df = run_to_dataframe(
    f"""
    SELECT
      table_catalog AS project,
      table_schema  AS dataset,
      table_name    AS table,
      ROUND(total_logical_bytes / 1024, 1) AS kb
    FROM `{PROJECT_ID}.raw.INFORMATION_SCHEMA.TABLE_STORAGE`
    WHERE table_schema IN ('raw', 'curated', 'mart')
    ORDER BY dataset, table
    """
)
display(df)

## 10. Pipeline cost control

Every pattern above should be measured. The dry run tells you the cost before
you spend it.

In [ ]:
PIPELINE_QUERIES = {
    "raw -> curated (replace part)": f"""
        SELECT * FROM `{PROJECT_ID}.raw.orders_raw$20260115`
    """,
    "curated full scan (bad)": f"""
        SELECT * FROM `{PROJECT_ID}.curated.orders_part`
    """,
    "mart rebuild": f"""
        SELECT order_date, region, channel, COUNT(*), SUM(total_amount)
        FROM `{PROJECT_ID}.curated.orders_dedup`
        GROUP BY order_date, region, channel
    """,
    "dashboard from MV": f"""
        SELECT region, SUM(revenue) FROM `{PROJECT_ID}.mart.mv_daily_revenue`
        WHERE order_date = DATE '2026-01-15' GROUP BY region
    """,
}

print(f"{'step':<30} {'bytes':>14} {'est. cost':>12}")
print("-" * 60)
for name, sql in PIPELINE_QUERIES.items():
    est = estimate_cost(sql)
    print(f"{name:<30} {est['bytes_processed']:>14,} ${est['estimated_cost_usd']:>11.6f}")

print("\nThe 'curated full scan (bad)' line is the one to watch —")
print("a pipeline that forgets its partition filter costs 100x more.")

## 🔑 Takeaways

- Three layers: **raw → curated → mart**, partitioned by the same business date.
- **Idempotency is non-negotiable.** `REPLACE PARTITION` for full days, `MERGE`
  for incremental corrections, anti-join `INSERT` for append-only.
- **Deduplicate on a business key**, keeping the latest by timestamp.
- **DQ gates are queries** that must return zero offending rows; fail the run
  when they do not.
- **Marts are fully replaced**; they are derived, so history is pointless.
- **Serve dashboards from a materialized view**, not from the raw mart.
- Streaming needs **explicit `row_ids`** for idempotency and a **lookback window**
  for late arrivals.
- Every task in the DAG must be safe to retry.
- **Dry-run every pipeline query** — one missing partition filter is 100x.

---

## 📚 Next

Back to the [course index](../README.md) or read the chapter
[48 · Airflow and Composer DAGs](../chapters/48-airflow-composer-dag.md).